# HYOPE 감정 분류기 학습 노트북

KoELECTRA-v3를 AI Hub **노년층 대상 감성 분류** 데이터로 파인튜닝해
6종 감정(분노/기쁨/불안/당황/슬픔/상처) 분류기를 만든다.
학습된 모델은 `hyope-ai-server`에서 OpenAI(GPT) 분석 점수를 보정하는 참고용 플래깅에 쓰인다
(`AI_model_research.md` 참고).

## 워크플로우

1. **데이터 로딩 & EDA** — AI Hub + KOTE + KorEmpatheticDialogues, 레이블 분포 확인
2. **분할** — AI Hub에서 보류 테스트셋 10%를 먼저 떼고, 나머지 + 공개 데이터로 학습/검증
3. **학습** — `train/` 모듈 재사용, RTX 3070 Ti 기준 fp16
4. **평가** — 보류 테스트셋 분류 리포트 + 혼동 행렬 + 오류 분석
5. **추론 데모** — 노인 발화 예시 + 무레이블 test.csv 예측

## 알아둘 것

- `emotionClassification_test.zip`의 `test.csv`는 **레이블이 없다** (제출용 분할).
  그래서 채점 가능한 테스트셋은 train.csv에서 계층화 분할로 직접 만든다.
- 보류셋은 **AI Hub 데이터에서만** 뗀다 — 우리가 재고 싶은 건 노인 발화 도메인 성능이고,
  KOTE/KorEmpatheticDialogues는 학습 보강용이기 때문.
- `train/train.py`의 CLI 학습은 AI Hub 전체를 쓰므로, 그걸로 학습한 모델을
  보류셋으로 평가하면 점수가 부풀려진다. 정직한 평가는 이 노트북 방식으로 학습했을 때만 유효.

## 사전 준비

```bash
pip install -r ../requirements.txt
pip install jupyter matplotlib pandas   # 노트북 전용 의존성
```

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# train/ 모듈 임포트 경로 (노트북은 notebooks/ 에서 실행된다고 가정)
TRAIN_DIR = Path.cwd().parent / "train"
sys.path.insert(0, str(TRAIN_DIR))

from config import ID2LABEL, LABEL2ID, LABELS, TrainConfig
from data_loader import load_aihub_data, load_empathetic_data, load_kote_data
from evaluate import split_aihub_holdout
from train import EmotionDataset, compute_metrics

config = TrainConfig()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)

print(f"device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"레이블: {LABELS}")

## 1. 데이터 로딩

세 소스를 따로 로딩해 규모를 확인한다.

| 소스 | 성격 | 용도 |
|---|---|---|
| AI Hub 노년층 감성 분류 | 노인-챗봇 대화, 6종 레이블 원본 | 핵심 학습 데이터 + 보류 테스트셋 |
| KOTE | 44종 → 6종 매핑 | 학습 보강 |
| KorEmpatheticDialogues | 32종 → 6종 매핑 | 학습 보강 |

In [ ]:
aihub_samples = load_aihub_data(config)
kote_samples = load_kote_data() if config.use_kote else []
empathetic_samples = load_empathetic_data() if config.use_empathetic else []

sources = {
    "AI Hub": aihub_samples,
    "KOTE": kote_samples,
    "KorEmpathetic": empathetic_samples,
}

## 2. EDA — 레이블 분포와 텍스트 길이

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 한글 폰트 (Windows: Malgun Gothic / WSL·Linux: NanumGothic → sudo apt install fonts-nanum)
_available = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _font in ["Malgun Gothic", "NanumGothic", "NanumBarunGothic", "AppleGothic"]:
    if _font in _available:
        plt.rcParams["font.family"] = _font
        break
else:
    print("경고: 한글 폰트를 찾지 못했습니다. 차트의 한글이 깨질 수 있습니다.")
plt.rcParams["axes.unicode_minus"] = False

# 차트 팔레트 — 단일 변수는 blue 하나, 소스 구분만 카테고리 3색 고정 순서
BLUE, AQUA, YELLOW = "#2a78d6", "#1baf7a", "#eda100"
SOURCE_COLORS = {"AI Hub": BLUE, "KOTE": AQUA, "KorEmpathetic": YELLOW}
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
BLUE_RAMP = LinearSegmentedColormap.from_list(
    "blue_ramp", [SURFACE, "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)


def style_axes(ax):
    ax.set_facecolor(SURFACE)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, labelcolor=INK_2)

In [ ]:
# 레이블 × 소스 건수 표 (차트의 표 뷰 역할도 겸함)
count_df = pd.DataFrame(
    {name: pd.Series([s["label"] for s in samples]).value_counts() for name, samples in sources.items()}
).fillna(0).astype(int)
count_df.index = [ID2LABEL[i] for i in count_df.index]
count_df = count_df.reindex(LABELS)
count_df["합계"] = count_df.sum(axis=1)
display(count_df)

# 소스별 누적 가로 막대 — 감정별 데이터 규모와 소스 구성을 한눈에
order = count_df["합계"].sort_values().index
fig, ax = plt.subplots(figsize=(8, 3.6))
left = np.zeros(len(order))
for name in sources:
    vals = count_df.loc[order, name].values
    ax.barh(order, vals, left=left, height=0.55, color=SOURCE_COLORS[name],
            label=name, edgecolor="white", linewidth=2)
    left += vals
for i, total in enumerate(count_df.loc[order, "합계"]):
    ax.text(total + left.max() * 0.01, i, f"{total:,}", va="center", fontsize=9, color=INK_2)
style_axes(ax)
ax.set_title("감정 레이블 분포 (소스별)", color=INK, loc="left", fontsize=12)
ax.legend(frameon=False, loc="lower right", labelcolor=INK_2)
ax.set_xlim(0, left.max() * 1.12)
plt.tight_layout()
plt.show()

In [ ]:
# 텍스트 길이 분포 — max_length(256 토큰) 설정이 적절한지 확인
all_samples = aihub_samples + kote_samples + empathetic_samples
lengths = pd.Series([len(s["text"]) for s in all_samples])

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lengths.clip(upper=400), bins=50, color=BLUE, edgecolor="white", linewidth=0.5)
style_axes(ax)
ax.set_title("텍스트 길이 분포 (글자 수, 400자에서 절단)", color=INK, loc="left", fontsize=12)
ax.grid(axis="y", color=GRID, linewidth=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

print(lengths.describe().round(1))
print(f"256자 초과 비율: {(lengths > 256).mean():.1%}")

## 3. 데이터 분할

```
AI Hub ──┬── 90% ──┐
         └── 10% ──┼──────────────→ 보류 테스트셋 (학습에 절대 사용 안 함)
KOTE ──────────────┤
KorEmpathetic ─────┴── 학습 풀 ──┬── 90% → 학습셋
                                 └── 10% → 검증셋 (early stopping 기준)
```

분할 로직은 `train/evaluate.py`의 `split_aihub_holdout()`과 공유한다
(같은 seed → 같은 보류셋, CLI 평가와 노트북 평가가 일치).

In [ ]:
from sklearn.model_selection import train_test_split

aihub_rest, test_samples = split_aihub_holdout(config)

pool = aihub_rest + kote_samples + empathetic_samples
train_samples, val_samples = train_test_split(
    pool,
    test_size=config.val_ratio,
    random_state=config.seed,
    stratify=[s["label"] for s in pool],
)

print(f"학습: {len(train_samples):,}건 / 검증: {len(val_samples):,}건 / 보류 테스트: {len(test_samples):,}건")

## 4. 학습

`train/train.py`와 같은 하이퍼파라미터에 fp16만 추가 (RTX 3070 Ti 기준 속도 ~2배).
저장 위치도 동일한 `model_output/` — `predict.py`와 FastAPI 통합 코드가 그대로 쓸 수 있다.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

tokenizer = AutoTokenizer.from_pretrained(config.model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

train_dataset = EmotionDataset(train_samples, tokenizer, config.max_length)
val_dataset = EmotionDataset(val_samples, tokenizer, config.max_length)

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    weight_decay=config.weight_decay,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=(DEVICE == "cuda"),
    logging_steps=100,
    seed=config.seed,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config.early_stopping_patience)],
)

trainer.train()
trainer.save_model(config.output_dir)
tokenizer.save_pretrained(config.output_dir)
print(f"모델 저장 완료: {config.output_dir}")

## 5. 평가 — 보류 테스트셋

학습에 한 번도 쓰이지 않은 AI Hub 보류셋으로 채점한다.
CLI로도 같은 평가 가능: `cd ../train && python evaluate.py`

In [ ]:
from evaluate import evaluate

eval_result = evaluate(config.output_dir, config)

In [ ]:
# 혼동 행렬 — 행 정규화(재현율 관점), 셀에 건수 표기
cm = eval_result["confusion"]
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6.4, 5.6))
im = ax.imshow(cm_norm, cmap=BLUE_RAMP, vmin=0, vmax=1)
ax.set_xticks(range(len(LABELS)), LABELS)
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("예측", color=INK_2)
ax.set_ylabel("정답", color=INK_2)
ax.tick_params(colors=MUTED, labelcolor=INK_2)
ax.set_title("혼동 행렬 (행 정규화)", color=INK, loc="left", fontsize=12)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        text_color = "white" if cm_norm[i, j] > 0.5 else INK_2
        ax.text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]:.0%}",
                ha="center", va="center", fontsize=8.5, color=text_color)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# 오류 분석 — 모델이 높은 확신으로 틀린 사례부터 (레이블 매핑 문제나 데이터 노이즈 힌트)
err_df = pd.DataFrame({
    "텍스트": eval_result["texts"],
    "정답": [ID2LABEL[t] for t in eval_result["true"]],
    "예측": [r["emotion"] for r in eval_result["results"]],
    "확신도": [r["confidence"] for r in eval_result["results"]],
})
errors = err_df[err_df["정답"] != err_df["예측"]].sort_values("확신도", ascending=False)

print(f"오분류 {len(errors)}건 / 전체 {len(err_df)}건 ({len(errors) / len(err_df):.1%})")
with pd.option_context("display.max_colwidth", 80):
    display(errors.head(15))

# 가장 자주 헷갈리는 정답→예측 쌍
print(errors.groupby(["정답", "예측"]).size().sort_values(ascending=False).head(8))

## 6. 추론 데모

FastAPI 통합 시 사용할 `EmotionPredictor` 그대로 사용.

In [ ]:
from predict import EmotionPredictor

predictor = EmotionPredictor(config.output_dir, device=DEVICE)

demo_texts = [
    "요즘 너무 외롭고 힘들어요. 아무것도 하기 싫어요.",
    "오늘 손주가 놀러와서 정말 기분이 좋았어요.",
    "약을 먹어야 하는데 자꾸 깜빡해서 걱정돼요.",
    "어젯밤에 잠을 통 못 잤어. 자꾸 이상한 소리가 들리는 것 같아서.",
    "아들이 전화 한 통을 안 해. 서운해 죽겠어.",
    "밥은 잘 먹었지. 반찬도 맛있게 해 먹었어.",
]
for text, result in zip(demo_texts, predictor.predict_batch(demo_texts)):
    top3 = sorted(result["scores"].items(), key=lambda x: -x[1])[:3]
    print(f"{result['emotion']} ({result['confidence']:.2f})  {text}")
    print(f"    상위 3: {', '.join(f'{k} {v:.2f}' for k, v in top3)}")

In [ ]:
# 무레이블 test.csv 예측 미리보기 (채점 불가, 정성 확인용)
import csv
import io
import zipfile

with zipfile.ZipFile(Path(config.aihub_train_zip).parent / "emotionClassification_test.zip") as z:
    with z.open("test.csv") as f:
        rows = list(csv.DictReader(io.StringIO(f.read().decode("utf-8-sig"))))

test_texts = [
    " ".join(row[f"사람문장{i}"].strip() for i in range(1, 5) if row.get(f"사람문장{i}", "").strip())
    for row in rows[:10]
]
for text, result in zip(test_texts, predictor.predict_batch(test_texts)):
    print(f"{result['emotion']} ({result['confidence']:.2f})  {text[:70]}")

## 다음 단계

1. **FastAPI 통합** — `hyope-ai-server`에 `model_output/`을 복사(또는 경로 참조)하고:

   ```python
   # hyope-ai-server 쪽에서
   from predict import EmotionPredictor
   emotion_predictor = EmotionPredictor("./model_output")  # CPU로도 동작
   # /analyze-call에서 OpenAI(GPT) 분석 점수 옆에 참고용 감정 플래그로 추가
   ```

2. **개선 실험 아이디어**
   - 클래스 불균형 대응: 소수 레이블 가중 loss 또는 KOTE/Empathetic 샘플링 비율 조정
   - `config.use_kote / use_empathetic` on/off 비교 — 공개 데이터 보강이 AI Hub 보류셋 성능에 실제 도움이 되는지 확인
   - AI Hub **생성형AI 고령자 구술 데이터**(dataSetSn=71703) 승인 후 우울·불안 회귀 태스크 추가

3. **EXAONE QLoRA** — 로컬 8GB VRAM으로는 빠듯하므로 Colab Pro / RunPod 확보 후 별도 노트북으로
   (`AI_model_research.md` §3~4의 설정값 참고)